In [1]:

import os
import getpass
import json
import altair as alt

import pandas as pd
from sklearn.manifold import TSNE
import umap
import numpy as np
from ast import literal_eval
# create interactive plot with gradio
import gradio as gr
from gradio.components import scatter_plot
import requests
from itertools import islice
import hdbscan
from sklearn import metrics
import matplotlib.pyplot as plt

reducer = umap.UMAP()
alt.data_transformers.disable_max_rows()


c:\Users\20195435\Documents\pyenv\nanopy310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DataTransformerRegistry.enable('default')

In [2]:

# load the embeddings from the json files
folder_path = 'embeddings'
json_files = [f for f in os.listdir(folder_path) if f.endswith('.json')]
# json_files = json_files[:1000]
data = {}

for file in json_files:
    with open(os.path.join(folder_path, file), 'r') as f:
        data[file] = json.load(f)

# data to dataframe
df = pd.DataFrame(data).T
df['size'] = 10
df['color'] = 'red'

# replace abstracts with the first abstract in the list if it is a list
df['abstract'] = df['abstract'].apply(lambda x: x[0] if isinstance(x, list) else x)

# filter the dataframe to only include journals from the following list
# List of journals to include
journals_to_include = [
    "ACS Applied Materials & Interfaces",
    "ACS Nano",
    "Advanced Functional Materials",
    "Advanced Materials",
    "Angewandte Chemie",
    "Biology and Medicine",
    "Biomaterials",
    "Cell",
    "Clinical Cancer Research",
    "Frontiers in Nanotechnology",
    "Immunity",
    "International Journal of Nanomedicine",
    "Journal of Controlled Release",
    "Journal of Materials Chemistry B",
    "Matter",
    "Molecular Therapy",
    "Nano Letters",
    "Nano Micro Small",
    "Nano Research",
    "Nanomedicine",
    "Nanomedicine: Nanotechnology",
    "Nanoscale",
    "Nature",
    "Nature Biomedical Engineering",
    "Nature Cancer",
    "Nature Communications",
    "Nature Materials",
    "Nature Medicine",
    "Nature Nanotechnology",
    "NPG Asia Materials",
    "Pharmaceutics",
    "PNAS",
    "Science",
    "Science Advances",
    "Science Translational Medicine",
    "Scientific Reports",
    "Small"
]

# journals as small letters
journals_to_include = [journal.lower() for journal in journals_to_include]

# Exclusion criteria
keywords_exclusion = ["review", "not available"]

# filter the dataframe to only include journals from the list
df = df[df['journal'].str.lower().isin(journals_to_include)]

# filter the dataframe to exclude titles with keywords from the exclusion list
df = df[~df['title'].str.lower().str.contains('|'.join(keywords_exclusion))]

# store the embeddings in a numpy array
embeddings = np.array(df['embedding'].map(lambda x: np.array(x)))
embeddings = np.stack(embeddings)


KeyboardInterrupt: 

In [ ]:
# create a t-SNE object
tsne = TSNE(n_components=2, random_state=42, init='random', learning_rate=200, max_iter=1000)

# fit the t-SNE object to the embeddings
tsne_embeddings = tsne.fit_transform(np.stack(embeddings[:]))

# create new pandas df with old df added the tsne embeddings
df_tsne = df.copy()

df_tsne['tsne_x'] = tsne_embeddings[:, 0]
df_tsne['tsne_y'] = tsne_embeddings[:, 1]

